In [1]:
import torch
import torch.nn.functional as F

def quantify_stft_boundary_discontinuity():
    """
    Quantifies C0 discontinuity (Step Jump Delta) at boundary frames
    and measures corresponding Spectral Energy Leakage in STFT.
    """
    # 1. Synthetic Signal with Non-Zero End boundary value (Value = 0.8 at border)
    t = torch.linspace(0, 10 * 3.14159, 1000)
    raw_signal = torch.sin(t) + 0.8
    pad_len = 128

    # 2. Apply Constant (0-Padding) vs Reflect Padding
    padded_const = F.pad(raw_signal, (pad_len, pad_len), mode='constant', value=0.0)
    padded_reflect = F.pad(raw_signal.unsqueeze(0).unsqueeze(0), (pad_len, pad_len), mode='reflect').squeeze()

    # 3. Calculate Boundary Step Jump Delta (|x[border] - x[border-1]|)
    jump_const = torch.abs(padded_const[pad_len] - padded_const[pad_len - 1]).item()
    jump_reflect = torch.abs(padded_reflect[pad_len] - padded_reflect[pad_len - 1]).item()

    # 4. Measure High-Frequency Spectral Leakage via STFT
    n_fft = 256
    window = torch.hann_window(n_fft)

    stft_const = torch.stft(padded_const, n_fft=n_fft, hop_length=64, win_length=n_fft, window=window, return_complex=True)
    stft_reflect = torch.stft(padded_reflect, n_fft=n_fft, hop_length=64, win_length=n_fft, window=window, return_complex=True)

    high_freq_leakage_const = torch.abs(stft_const)[n_fft // 4:, :].mean().item()
    high_freq_leakage_reflect = torch.abs(stft_reflect)[n_fft // 4:, :].mean().item()

    return {
        "Constant Jump Delta": jump_const,
        "Reflect Jump Delta": jump_reflect,
        "Constant High-Freq Leakage": high_freq_leakage_const,
        "Reflect High-Freq Leakage": high_freq_leakage_reflect
    }

# Hypothesis Proof Metric
results = quantify_stft_boundary_discontinuity()
for key, val in results.items():
    print(f"{key:<30}: {val:.6f}")


Constant Jump Delta           : 0.800000
Reflect Jump Delta            : 0.031442
Constant High-Freq Leakage    : 0.089903
Reflect High-Freq Leakage     : 0.004968
